# 02 - Engenharia de Features

Notebook para criar variaveis derivadas, transformar atributos e preparar bases modelaveis.

## 1 — Carregamento e diagnóstico dos dados brutos

Parti da base oficial Home Credit (`application_train.csv`). Toda a lógica abaixo está encapsulada em `src/` — este notebook **demonstra e documenta** as transformações que o pipeline de treino aplica.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
from src.preprocessing import tratar_nulos
from src.features import (
    criar_features,
    criar_features_open_finance_simulado,
    codificar_categoricas,
)

df = pd.read_csv('../data/raw/application_train.csv')
print(f'Shape: {df.shape}')
print(f'Total de nulos: {df.isnull().sum().sum():,}')
print(f'Colunas categóricas (object): {(df.dtypes == "object").sum()}')
df.head()

## 2 — Tratamento de nulos

`tratar_nulos` imputa numéricas pela **mediana** (robusta a outliers) e categóricas por `'DESCONHECIDO'`. No pipeline real (notebook 03) a mediana é aprendida só no treino (`fit=True`) e reaplicada ao teste (`fit=False`), evitando leakage.

In [ ]:
nulos_antes = df.isnull().sum().sum()
df_clean, imputers = tratar_nulos(df, fit=True)  # fit aprende as medianas
print(f'Nulos antes: {nulos_antes:,}  ->  depois: {df_clean.isnull().sum().sum()}')
print(f'Medianas aprendidas para {len(imputers["medianas"])} colunas numéricas')

## 3 — Features de negócio (`criar_features`)

Variáveis derivadas com significado de crédito:
- **Capacidade de pagamento:** `RAZAO_ANUIDADE_RENDA`, `RAZAO_CREDITO_RENDA`, `PRAZO_ANOS`
- **Demografia:** `IDADE_ANOS`, `EMPREGO_ANOS`, `PROP_VIDA_TRABALHADA`
- **Scores externos:** `SCORE_EXTERNO_MEDIO`, `SCORE_EXTERNO_PROD`

In [ ]:
cols_antes = set(df_clean.columns)
df_feat = criar_features(df_clean)
novas = sorted(set(df_feat.columns) - cols_antes)
print('Novas features criadas:')
for c in novas:
    print(' -', c)

df_feat[novas].describe().T[['mean', 'std', 'min', 'max']].round(3)

## 4 — Features simuladas de Open Finance

O diferencial do projeto é incorporar sinais de **Open Finance**. Como não há dados transacionais reais disponíveis publicamente, `criar_features_open_finance_simulado` **simula** indicadores típicos: renda recorrente, comprometimento de renda, volatilidade de saldo, atrasos e um índice de estabilidade financeira.

⚠️ **Transparência:** por serem simuladas, algumas dessas features injetam sinal a partir do `TARGET` — servem para ilustrar a *forma* das variáveis de Open Finance e **não** são usadas no pipeline de treino sem leakage (notebook 03, que usa apenas `criar_features`). Num cenário real, viriam de extratos/transações via Open Finance.

In [ ]:
df_of = criar_features_open_finance_simulado(df_clean)
features_of = [
    'RENDA_RECORRENTE_MEDIA_6M',
    'COMPROMETIMENTO_RENDA_ESTIMADO',
    'VOLATILIDADE_SALDO_6M',
    'ATRASOS_PAGAMENTO_12M',
    'INDICE_ESTABILIDADE_FINANCEIRA',
]
df_of[features_of].describe().T[['mean', 'std', 'min', 'max']].round(3)

## 5 — Codificação de variáveis categóricas

`codificar_categoricas` aplica **Label Encoding** (o LightGBM lida bem com categóricas inteiras) e devolve os `encoders` para reaplicar em produção. Categorias não vistas no treino recebem um código próprio, sem quebrar o pipeline.

In [ ]:
obj_antes = (df_feat.dtypes == 'object').sum()
df_enc, encoders = codificar_categoricas(df_feat, fit=True)
print(f'Colunas object antes: {obj_antes}  ->  depois: {(df_enc.dtypes == "object").sum()}')
print(f'Encoders guardados: {len(encoders)} colunas')
print('Exemplo (CODE_GENDER):', list(encoders['CODE_GENDER'].classes_))

## 6 — Conclusão

A engenharia de features está encapsulada em `src/features.py` e `src/preprocessing.py` (funções testadas, reaproveitadas pelos notebooks 03–05). Isso garante que **treino e produção usem exatamente a mesma transformação** — pré-requisito para um modelo de crédito confiável. O notebook 03 consome essas funções no fluxo sem leakage (fit no treino, transform no teste).